# 02 - Image Preprocessing Pipeline

Milestone 2: Preparing the Rakuten product images for reproducible image and multimodal modelling.

This notebook converts the image-audit findings into a reusable preprocessing workflow. It does not train an image model. Raw files remain unchanged, and images are opened only when they are needed.


## 1. Scope and Audit Context

The data-audit notebooks established the following points:

- all 84,916 labelled training products have a matching image file;
- all 2,000 sampled images were readable, RGB images with a size of 500 × 500 pixels;
- exact file duplicates were audited for the labelled training catalogue only;
- the portable training-image file hashes were cached under `data/processed/`.

The purpose of this notebook is therefore not to repeat the image audit. It prepares the image manifest, loads the shared validation split created by `02_preprocessing.ipynb`, uses the cached audit metadata to create a cleaned image training set while keeping validation unchanged, and defines deterministic framework-neutral image preprocessing.


## 2. Imports and Configuration

Project paths and image filenames remain centralised in `src.data_utils`. No dataset path is duplicated in this notebook.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

from src.data_utils import (
    get_data_paths,
    get_image_path,
    load_image,
    load_merged_data,
)

RANDOM_STATE = 42
IMAGE_SIZE = (224, 224)

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Environment and data utilities loaded successfully.")


Environment and data utilities loaded successfully.


## 3. Build the Training Image Manifest

`load_merged_data()` returns the labelled challenge data. The original dataframe index is retained as `row_id`; this is the safest key for joining text predictions, image predictions and split assignments later in the project.

The manifest stores metadata and file paths only. It does not load 84,916 images into memory.


In [2]:
data_paths = get_data_paths()

train_df = (load_merged_data().rename_axis("row_id").reset_index())

train_images                    = train_df[["row_id", "imageid", "productid", "prdtypecode"]].copy()
train_images["source_split"]    = "train"
train_images["prdtypecode"]     = train_images["prdtypecode"].astype("Int64")

# A list comprehension is faster than a row-wise DataFrame.apply for this task.
train_images["image_path"] = [
    Path(
        get_image_path(
            int(row.imageid),
            int(row.productid),
            split="image_train",
        )
    )
    for row in train_images.itertuples(index=False)
]

print("Training image rows:", f"{len(train_images):,}")
display(train_images.head())


Training image rows: 84,916


,row_id,imageid,productid,prdtypecode,source_split,image_path
0,0,1263597046,3804725264,10,train,/Users/Ben/Development/VS-Python/Rakuten2/lior...
1,1,1008141237,436067568,2280,train,/Users/Ben/Development/VS-Python/Rakuten2/lior...
2,2,938777978,201115110,50,train,/Users/Ben/Development/VS-Python/Rakuten2/lior...
3,3,457047496,50418756,1280,train,/Users/Ben/Development/VS-Python/Rakuten2/lior...
4,4,1077757786,278535884,2705,train,/Users/Ben/Development/VS-Python/Rakuten2/lior...


## 4. Validate the Manifest

The earlier audit already checked the complete training filename coverage. These assertions protect the preprocessing workflow against accidental path, join or data changes.


In [3]:
assert train_images["row_id"].is_unique
assert train_images["imageid"].notna().all()
assert train_images["productid"].notna().all()
assert train_images["prdtypecode"].notna().all()

train_images["image_exists"]    = train_images      ["image_path"   ].map(Path.exists)
available_images                = int(train_images  ["image_exists" ].sum())
missing_images                  = int((~train_images["image_exists" ]).sum())

print(f"Available training images: {available_images:,}/{len(train_images):,}")
print(f"Missing training images  : {missing_images:,}")

assert train_images["image_exists"].all(), "At least one expected training image is missing."


Available training images: 84,916/84,916
Missing training images  : 0


## 5. Load the Shared Training/Validation Split

The official challenge test data has no labels and is not used for local evaluation. This notebook therefore loads the shared training/validation assignment created from the labelled catalogue in `02_preprocessing.ipynb`. The assignment remains unchanged here so text and image predictions can still be joined through `row_id`.


In [4]:
split_path = (
    Path(data_paths["x_train"]).parent.parent
    / "processed"
    / "data_split.csv"
)

if not split_path.exists():
    raise FileNotFoundError(
        "Shared split manifest not found. "
        "Run notebooks/02_preprocessing.ipynb before this notebook."
    )

split_manifest = pd.read_csv(split_path, usecols=["row_id", "split"])

# Reject stale, incomplete or ambiguous split files before using them.
assert split_manifest["row_id"].is_unique
assert split_manifest["split"].notna().all()
assert set(split_manifest["split"]) == {"train", "validation"}
assert set(split_manifest["row_id"]) == set(train_images["row_id"])

train_images = train_images.merge(
    split_manifest,
    on="row_id",
    how="left",
    validate="one_to_one",
)
assert train_images["split"].notna().all()

split_summary = (
    train_images.groupby("split")
    .agg(
        products=("row_id", "size"),
        classes=("prdtypecode", "nunique"),
    )
)
split_summary["share"] = split_summary["products"] / len(train_images)

print(f"Loaded shared split manifest: {split_path}")
display(split_summary.style.format({"share": "{:.1%}"}))


Loaded shared split manifest: /Users/Ben/Development/VS-Python/Rakuten2/liora-capstone/data/processed/data_split.csv


,products,classes,share
split,,,
train,67932,27,80.0%
validation,16984,27,20.0%


## 6. Create Duplicate- and Conflict-Free Image Sets

The cached file SHA-256 hashes identify exact image matches. The shared validation set remains completely unchanged so that text and image models are evaluated on exactly the same products. Cleaning is restricted to training and does not overwrite the shared split. The following deterministic rules are applied:

1. keep every row assigned to validation;
2. remove training rows whose file hash also occurs in validation to prevent leakage;
3. remove conflicting file-hash groups from training only;
4. keep one training row per remaining file hash and retain `row_id` for multimodal comparison.

The raw data and the shared `data_split.csv` remain unchanged.


In [5]:
hash_cache_path = (
    Path(data_paths["x_train"]).parent.parent
    / "processed"
    / "training_image_duplicate_hashes.json"
)

if not hash_cache_path.exists():
    raise FileNotFoundError(
        "Training-image hash cache not found. "
        "Run the image duplicate audit first."
    )

hash_payload = json.loads(hash_cache_path.read_text(encoding="utf-8"))
hash_records = hash_payload.get("images", {})

train_images["image_filename"] = train_images["image_path"    ].map(lambda path: Path(path).name)
train_images["file_sha256"   ] = train_images["image_filename"].map(lambda filename: hash_records.get(filename, {}).get("file_sha256") )

missing_hashes          = int(train_images["file_sha256"].isna().sum())
assert missing_hashes == 0, f"{missing_hashes} images have no file hash."

# Keep the complete shared validation set unchanged.
image_validation_set = (
    train_images.loc[train_images["split"].eq("validation")]
    .sort_values("row_id")
    .reset_index(drop=True)
)
expected_validation_ids = set(split_manifest.loc[split_manifest["split"].eq("validation"), "row_id"])

# Remove training images that also occur in validation to prevent leakage.
validation_hashes   = set(image_validation_set["file_sha256"])
training_candidates = train_images.loc[
    train_images["split"].eq("train")
    & ~train_images["file_sha256"].isin(validation_hashes)
].copy()

# Duplicate and label-conflict cleaning is restricted to training.
file_group_stats = (
    training_candidates.groupby("file_sha256")
    .agg(
        group_size  =("row_id", "size"),
        label_count =("prdtypecode", "nunique"),
    )
    .reset_index()
)

training_data = training_candidates.merge(
    file_group_stats,
    on          ="file_sha256",
    how         ="left",
    validate    ="many_to_one",
)

# Exact images associated with different labels are removed from training only.
conflict_mask      = training_data    ["label_count"].gt(1)
conflicting_images = training_data.loc[conflict_mask].copy()
eligible_training  = training_data.loc[~conflict_mask].copy()

image_training_set = (
    eligible_training
    .sort_values("row_id")
    .drop_duplicates("file_sha256", keep="first")
    .reset_index(drop=True)
)

excluded_images = train_images.loc[ train_images["split"].eq("train") & ~train_images["row_id"].isin(image_training_set["row_id"]) ].copy()
conflict_hashes = set(conflicting_images["file_sha256"])
excluded_images["exclusion_reason"] = np.select(
    [
        excluded_images["file_sha256"].isin(validation_hashes),
        excluded_images["file_sha256"].isin(conflict_hashes),
    ],
    ["validation_hash_overlap", "conflicting_labels"],
    default="duplicate_same_label",
)

# Model-facing integrity checks.
assert image_training_set["file_sha256"].is_unique
assert set(image_training_set["file_sha256"]).isdisjoint( set(image_validation_set["file_sha256"]))
assert len(image_validation_set)                        == len(expected_validation_ids)
assert set(image_validation_set["row_id"])              == expected_validation_ids
assert image_training_set["prdtypecode"].nunique()      == 27
assert image_validation_set["prdtypecode"].nunique()    == 27

cleaning_summary = pd.DataFrame({
    "before": [
        int(train_images["split"].eq("train").sum()),
        int(train_images["split"].eq("validation").sum()),
    ],
    "after": [len(image_training_set), len(image_validation_set)],
    "classes": [
        image_training_set["prdtypecode"].nunique(),
        image_validation_set["prdtypecode"].nunique(),
    ],
    "unique_file_hashes": [
        image_training_set["file_sha256"].nunique(),
        image_validation_set["file_sha256"].nunique(),
    ],
}, index=["train", "validation"])

cleaning_summary["removed"      ] = ( cleaning_summary["before" ] - cleaning_summary["after"] )
cleaning_summary["removed_share"] = ( cleaning_summary["removed"] / cleaning_summary["before"] )

train_class_counts      = image_training_set["prdtypecode"].value_counts().sort_index()
validation_class_counts = ( image_validation_set["prdtypecode"].value_counts().sort_index() )

clean_class_distribution = pd.DataFrame({ "train_count": train_class_counts, "validation_count": validation_class_counts })
clean_class_distribution["train_share"              ] = (clean_class_distribution["train_count"     ] / len(image_training_set))
clean_class_distribution["validation_share"         ] = (clean_class_distribution["validation_count"] / len(image_validation_set) )
clean_class_distribution["absolute_share_difference"] = (clean_class_distribution["train_share"     ] - clean_class_distribution["validation_share"]).abs()

display(cleaning_summary.style.format({"removed_share": "{:.2%}"}))
display(
    clean_class_distribution.style.format({
        "train_share": "{:.2%}",
        "validation_share": "{:.2%}",
        "absolute_share_difference": "{:.2%}",
    })
)
print(f"Excluded conflict rows: {len(conflicting_images):,}")
print(f"Excluded rows in total: {len(excluded_images):,}")


,before,after,classes,unique_file_hashes,removed,removed_share
train,67932,62312,27,62312,5620,8.27%
validation,16984,16984,27,16603,0,0.00%


,train_count,validation_count,train_share,validation_share,absolute_share_difference
prdtypecode,,,,,
10,2483,623,3.98%,3.67%,0.32%
40,1978,502,3.17%,2.96%,0.22%
50,1272,336,2.04%,1.98%,0.06%
60,623,166,1.00%,0.98%,0.02%
1140,2127,534,3.41%,3.14%,0.27%
1160,3135,791,5.03%,4.66%,0.37%
1180,605,153,0.97%,0.90%,0.07%
1280,3528,974,5.66%,5.73%,0.07%
1281,1601,414,2.57%,2.44%,0.13%


Excluded conflict rows: 676
Excluded rows in total: 5,620


## 7. Export the Clean Image Split

The model-facing manifest contains identifiers, labels, split assignments and file hashes only. Absolute image paths are not stored, so each machine can reconstruct them with `get_image_path()`.

In [6]:
manifest_columns = [
    "row_id",
    "imageid",
    "productid",
    "prdtypecode",
    "split",
    "file_sha256",
]

image_clean = (
    pd.concat([image_training_set, image_validation_set], ignore_index=True)
    .sort_values(["split", "row_id"])
    .reset_index(drop=True)
)

image_manifest_path = (
    Path(data_paths["x_train"]).parent.parent
    / "processed"
    / "image_split_clean.csv"
)
image_manifest_path.parent.mkdir(parents=True, exist_ok=True)
image_clean[manifest_columns].to_csv(image_manifest_path, index=False)

saved_manifest = pd.read_csv(image_manifest_path)
assert len(saved_manifest) == len(image_clean)
assert saved_manifest["row_id"].is_unique
assert set(saved_manifest["split"]) == {"train", "validation"}

print(f"Saved clean image manifest: {image_manifest_path}")
print(f"Rows: {len(saved_manifest):,}")


Saved clean image manifest: /Users/Ben/Development/VS-Python/Rakuten2/liora-capstone/data/processed/image_split_clean.csv
Rows: 79,296
